In [ ]:
import logging
import os

import pandas as pd

from scGPT import load_scgpt, load_gene_annotations, extract_model_weights, SCGPT_DEFS

from napistu.utils import download_wget

from etl_utils import (
    compute_attention_from_weights, 
    save_results,
    load_results,
    RESULTS_DEFS
)

logger = logging.getLogger(__name__)

/opt/homebrew/Caskroom/miniforge/base/envs/scgpt/lib/python3.11/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/opt/homebrew/Caskroom/miniforge/base/envs/scgpt/lib/python3.11/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/opt/homebrew/Caskroom/miniforge/base/envs/scgpt/lib/python3.11/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/opt/homebrew/Caskroom/miniforge/base/envs/scgpt/lib/python3.11/site-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TO

In [2]:
DATA_DIR = "data"
OUTPUT_DIR = "output"

MODEL_PATH = os.path.join(DATA_DIR, "scGPT_bc")
MODEL_RESULTS_PATH = os.path.join(OUTPUT_DIR, "scgpt_weights.npz")
ANNOTATIONS_PATH = os.path.join(DATA_DIR, "scgpt_gene_info.csv")

In [3]:
# download ensembl <-> aliases mappings
if not os.path.isfile(ANNOTATIONS_PATH):
    download_wget(SCGPT_DEFS.GENE_IDENTIFIERS_URL, ANNOTATIONS_PATH)

gene_annotations = load_gene_annotations(ANNOTATIONS_PATH)

logger.info("Loading scGPT model")
model, vocab, model_metadata = load_scgpt(MODEL_PATH)

logger.info("Extracting model weights")
weights_dict = extract_model_weights(model, vocab, model_metadata)

logger.info(f"Saving weights to {OUTPUT_DIR}")
save_results(weights_dict, gene_annotations, model_metadata, OUTPUT_DIR, SCGPT_DEFS.MODEL_NAME)

          ensembl_gene         symbol     vocab_name
0      ENSG00000121410           A1BG           A1BG
1      ENSG00000268895       A1BG-AS1       A1BG-AS1
2      ENSG00000148584           A1CF           A1CF
3      ENSG00000175899            A2M            A2M
4      ENSG00000245105        A2M-AS1        A2M-AS1
...                ...            ...            ...
60659  ENSG00000288719  RP4-669P10.21  RP4-669P10.21
60660  ENSG00000288720  RP11-852E15.3  RP11-852E15.3
60661  ENSG00000288721   RP5-973N23.5   RP5-973N23.5
60662  ENSG00000288723  RP11-553N16.6  RP11-553N16.6
60663  ENSG00000288724   RP13-546I2.2   RP13-546I2.2

[60664 rows x 3 columns]
Resume model from data/scGPT_bc/best_model.pt, the model args will override the config data/scGPT_bc/args.json.
Loading params encoder.embedding.weight with shape torch.Size([60697, 512])
Loading params encoder.enc_norm.weight with shape torch.Size([512])
Loading params encoder.enc_norm.bias with shape torch.Size([512])
Loading params v

In [4]:
weights_dict, gene_annotations, model_metadata = load_results(OUTPUT_DIR, SCGPT_DEFS.MODEL_NAME)

GENES_OF_INTEREST = gene_annotations[RESULTS_DEFS.VOCAB_NAME].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in model_metadata[RESULTS_DEFS.ORDERED_VOCABULARY]]

# Compute attention on demand
layer_11_attn = compute_attention_from_weights(
    weights_dict[RESULTS_DEFS.GENE_EMBEDDING][GENE_MASK,:],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_11'][RESULTS_DEFS.W_Q],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_11'][RESULTS_DEFS.W_K]
)